# Proyecto de Inteligencia Artificial — UPAO

**Sistema inteligente para la evaluación y predicción de la competitividad laboral de estudiantes universitarios mediante Inteligencia Artificial**

Notebook completo del proyecto, ordenado según las secciones del informe.

**Cómo ejecutarlo:** en un entorno nuevo de Google Colab, `Entorno de ejecución → Ejecutar todas`. La Celda 1 instala una versión fija de scikit-learn y la Celda 2 pide subir `graduate_employment_dataset.csv`.

# Secciones 3 y 4 — Data-Set, pre-procesamiento y aprendizaje

## Celda 1 — Librerías

In [ ]:
!pip install -q scikit-learn==1.8.0

import copy
import os
import pickle
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score,
                             classification_report, confusion_matrix, f1_score, log_loss,
                             matthews_corrcoef, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
OUT = 'salidas'
os.makedirs(OUT, exist_ok=True)
CSV = 'graduate_employment_dataset.csv'
AZUL, NARANJA, VERDE, ROJO = '#2E6F9E', '#E07B39', '#4C9A6A', '#C0504D'
ORDEN = ['Low', 'Medium', 'High']
PAL = {'Low': ROJO, 'Medium': NARANJA, 'High': VERDE}
print('Librerías cargadas')

## Celda 2 — Carga del Data-Set (3.1)

In [ ]:
if not os.path.exists(CSV):
    from google.colab import files
    files.upload()

df = pd.read_csv(CSV)

print('Dimensiones del dataset (filas, columnas):', df.shape)
print('\n--- Primeras filas ---')
print(df.head())
print('\n--- Tipos de dato y no-nulos ---')
df.info()

shape_original = df.shape

fig, ax = plt.subplots(figsize=(16, 3.1))
ax.axis('off')
sub = df.head(6)
encabezados = [c.replace('_', '\n') for c in sub.columns]
tb = ax.table(cellText=sub.values, colLabels=encabezados,
              cellLoc='center', loc='center')
tb.auto_set_font_size(False); tb.set_fontsize(7.0); tb.scale(1, 2.1)
for j in range(len(sub.columns)):
    c = tb[0, j]; c.set_facecolor('#1F3B57'); c.set_text_props(color='white', weight='bold')
for i in range(1, len(sub) + 1):
    for j in range(len(sub.columns)):
        if i % 2 == 0:
            tb[i, j].set_facecolor('#F2F5F8')
ax.set_title(f'Estructura del Data-Set: {df.shape[0]} registros x {df.shape[1]} variables',
             fontsize=10, fontweight='bold', pad=14)
fig.savefig(f'{OUT}/fig3_estructura.png'); plt.show()

## Celda 3 — Verificación de integridad (3.1.1-a)

In [ ]:
print('--- Control de calidad del Data-Set ---')
integridad = pd.DataFrame({
    'Tipo': df.dtypes.astype(str),
    'Nulos': df.isna().sum(),
    'Valores_unicos': df.nunique(),
    'Minimo': [df[c].min() if pd.api.types.is_numeric_dtype(df[c]) else '-' for c in df.columns],
    'Maximo': [df[c].max() if pd.api.types.is_numeric_dtype(df[c]) else '-' for c in df.columns],
})
print(integridad)
integridad.to_csv(f'{OUT}/tabla_integridad.csv')

print(f'\nFilas duplicadas          : {df.duplicated().sum()}')
print(f'Student_No duplicados     : {df["Student_No"].duplicated().sum()}')
print(f'Total de celdas nulas     : {df.isna().sum().sum()}')

## Celda 4 — Variable objetivo y desbalance de clases (3.1.1-b)

In [ ]:
print('--- Estadisticos de Employment_Competitiveness (puntaje continuo) ---')
print(df['Employment_Competitiveness'].describe())

conteo = df['Employment_Class'].value_counts().reindex(ORDEN)
porc = (conteo / len(df) * 100).round(2)
print('\n--- Distribucion de Employment_Class ---')
print(pd.DataFrame({'Registros': conteo, '%': porc}))

q1, q3 = df['Employment_Competitiveness'].quantile([.25, .75])
iqr = q3 - q1
li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_out = ((df['Employment_Competitiveness'] < li) |
         (df['Employment_Competitiveness'] > ls)).sum()
print(f'\nLimites IQR: [{li:.2f} , {ls:.2f}]  ->  outliers: {n_out} '
      f'({n_out/len(df)*100:.2f}%)')

print('\n--- Rango del puntaje por clase ---')
print(df.groupby('Employment_Class')['Employment_Competitiveness']
        .agg(['min', 'max', 'count']).reindex(ORDEN))

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4))
axes[0].hist(df['Employment_Competitiveness'], bins=45, color=AZUL, edgecolor='white')
for u in (40, 70):
    axes[0].axvline(u, color=ROJO, ls='--', lw=1.5)
axes[0].text(40, axes[0].get_ylim()[1]*0.94, ' 40', color=ROJO, fontsize=8, fontweight='bold')
axes[0].text(70, axes[0].get_ylim()[1]*0.94, ' 70', color=ROJO, fontsize=8, fontweight='bold')
axes[0].set_title('Puntaje de competitividad\ny umbrales de clase', fontsize=10)
axes[0].set_xlabel('Employment_Competitiveness'); axes[0].set_ylabel('Frecuencia')

sns.boxplot(x=df['Employment_Competitiveness'], ax=axes[1], color=NARANJA)
axes[1].set_title(f'Valores atipicos (regla IQR)\noutliers detectados: {n_out}', fontsize=10)
axes[1].set_xlabel('Employment_Competitiveness')

b = axes[2].bar([str(c) for c in ORDEN], conteo.values,
                color=[PAL[c] for c in ORDEN])
axes[2].set_title('Desbalance de Employment_Class', fontsize=10)
axes[2].set_ylabel('N.º de registros')
for r, v, pc in zip(b, conteo.values, porc.values):
    axes[2].text(r.get_x() + r.get_width()/2, v + 25, f'{v}\n({pc}%)',
                 ha='center', fontsize=8.5, fontweight='bold')
axes[2].set_ylim(0, conteo.max() * 1.22)
fig.suptitle('Análisis de la variable objetivo', y=1.03, fontsize=12, fontweight='bold')
fig.savefig(f'{OUT}/fig4_objetivo.png'); plt.show()

## Celda 5 — Correlaciones y selección de variables (3.1.1-c)

In [ ]:
num = [c for c in df.columns if c not in ('Student_No', 'Employment_Class')]
corr = df[num].corr()

fig, ax = plt.subplots(figsize=(9.5, 7.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            annot_kws={'size': 7}, cbar_kws={'label': 'Coef. de Pearson'}, ax=ax)
ax.set_title('Matriz de correlación entre variables del perfil y la competitividad',
             fontsize=11, fontweight='bold')
fig.savefig(f'{OUT}/fig5_correlacion.png'); plt.show()

rel = (corr['Employment_Competitiveness'].drop('Employment_Competitiveness')
       .sort_values(ascending=False))
print('--- Correlacion con Employment_Competitiveness ---')
print(rel.round(4))

UMBRAL = 0.10
irrelevantes = rel[rel.abs() < UMBRAL].index.tolist()
relevantes = rel[rel.abs() >= UMBRAL].index.tolist()

PRED = [c for c in num if c != 'Employment_Competitiveness']
y_cls = df['Employment_Class'].map({'Low': 0, 'Medium': 1, 'High': 2})

mi = pd.Series(mutual_info_classif(df[PRED], y_cls, random_state=42), index=PRED)
rf_sel = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                random_state=42, n_jobs=-1).fit(df[PRED], y_cls)
imp = pd.Series(rf_sel.feature_importances_, index=PRED)

criterios = pd.DataFrame({'Pearson_|r|': rel.abs().round(4),
                          'Informacion_mutua': mi.round(4),
                          'Importancia_RF': imp.round(4)}
                         ).sort_values('Informacion_mutua', ascending=False)
print('\n--- TABLA: evidencia convergente para la selección de variables ---')
print(criterios.to_string())
criterios.to_csv(f'{OUT}/tabla_criterios.csv')

print(f'\nVariables PREDICTORAS (|r| >= {UMBRAL}): {relevantes}')
print(f'Variables IRRELEVANTES (|r| < {UMBRAL}): {irrelevantes}')

skf_sel = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for etq, cols in [('con las 11 variables', PRED),
                  ('solo las 6 conservadas', relevantes)]:
    sc = cross_val_score(RandomForestClassifier(n_estimators=300,
                                                class_weight='balanced',
                                                random_state=42, n_jobs=-1),
                         df[cols], y_cls, cv=skf_sel, scoring='f1_macro')
    print(f'  F1-macro {etq:24s} = {sc.mean():.4f}  (desv. {sc.std():.4f})')

fig, ax = plt.subplots(figsize=(8.5, 4.6))
cols = [VERDE if abs(v) >= UMBRAL else '#B9C0C8' for v in rel.values]
ax.barh(rel.index[::-1], rel.values[::-1], color=cols[::-1])
ax.axvline(UMBRAL, color=ROJO, ls='--', lw=1.2)
ax.axvline(-UMBRAL, color=ROJO, ls='--', lw=1.2)
ax.set_xlabel('Correlación de Pearson con el puntaje de competitividad')
ax.set_title('Variables con y sin poder predictivo', fontsize=11, fontweight='bold')
for i, v in enumerate(rel.values[::-1]):
    ax.text(v + (0.012 if v >= 0 else -0.012), i, f'{v:.3f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=7.5)
ax.set_xlim(-0.12, 0.62)
fig.savefig(f'{OUT}/fig6_relevancia.png'); plt.show()

## Celda 6 — Perfil según nivel de competitividad (3.1.1-d)

In [ ]:
dims = [('GPA', 'Rendimiento académico'), ('Programming_Skill', 'Habilidad técnica'),
        ('Communication_Skill', 'Habilidad blanda'), ('Industry_Exposure', 'Experiencia')]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (c, t) in zip(axes, dims):
    sns.boxplot(data=df, x='Employment_Class', y=c, order=ORDEN,
                palette=[PAL[o] for o in ORDEN], ax=ax)
    ax.set_title(f'{t}\n({c})', fontsize=9.5)
    ax.set_xlabel(''); ax.set_ylabel('')
fig.suptitle('Perfil del estudiante según su nivel de competitividad laboral',
             y=1.04, fontsize=12, fontweight='bold')
fig.savefig(f'{OUT}/fig8_perfil.png'); plt.show()

print('--- Media de cada dimension por clase ---')
resumen = df.groupby('Employment_Class')[
    ['GPA', 'Programming_Skill', 'Communication_Skill', 'Teamwork',
     'Internship_Count', 'Industry_Exposure']].mean().reindex(ORDEN).round(2)
print(resumen)
resumen.to_csv(f'{OUT}/tabla_perfil_clase.csv')

## Celda 7 — Limpieza y filtrado (3.2)

In [ ]:
d = df.copy()
log = []
n0 = len(d)

d = d.drop(columns=['Student_No'])
log.append(('Eliminación del identificador', 'Student_No', d.shape[1]))

d = d.drop(columns=['Employment_Competitiveness'])
log.append(('Prevención de fuga de información',
            'Employment_Competitiveness (origen de la etiqueta)', d.shape[1]))

rangos = {'GPA': (0, 10), 'Programming_Skill': (1, 10), 'Communication_Skill': (1, 10),
          'Teamwork': (1, 10), 'Career_Motivation': (1, 10), 'Industry_Exposure': (0, 10),
          'Internship_Count': (0, 10), 'Age': (16, 70)}
fuera = {c: int(((d[c] < a) | (d[c] > b)).sum()) for c, (a, b) in rangos.items() if c in d}
print('Registros fuera de rango por variable:', fuera)
log.append(('Verificación de rangos válidos',
            f'{sum(fuera.values())} registros fuera de rango', d.shape[1]))

log.append(('Tratamiento de nulos', f'{df.isna().sum().sum()} nulos: no se requiere imputación',
            d.shape[1]))
log.append(('Eliminación de duplicados', f'{df.duplicated().sum()} duplicados encontrados',
            d.shape[1]))

d = d.drop(columns=[c for c in irrelevantes if c in d.columns])
log.append((f'Eliminación de variables irrelevantes (|r| < {UMBRAL})',
            ', '.join(irrelevantes), d.shape[1]))

print('\n--- TABLA: bitácora de preprocesamiento ---')
bit = pd.DataFrame(log, columns=['Operación', 'Detalle', 'Variables restantes'])
print(bit.to_string(index=False))
bit.to_csv(f'{OUT}/tabla_limpieza.csv', index=False)
print(f'\nRegistros: {n0} -> {len(d)}   |   Variables: {shape_original[1]} -> {d.shape[1]}')

## Celda 8 — Codificación y partición del Data-Set (3.3)

In [ ]:
y_txt = d['Employment_Class']
X = d.drop(columns=['Employment_Class'])

MAPA = {'Low': 0, 'Medium': 1, 'High': 2}
INV = {v: k for k, v in MAPA.items()}
y = y_txt.map(MAPA).rename('Employment_Class')
print('Codificación ordinal de la etiqueta:', MAPA)

cat_cols = list(X.select_dtypes(include=['object', 'string', 'category']).columns)
if cat_cols:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)
    print('Variables categóricas codificadas (One-Hot):', cat_cols)
else:
    print('Variables categóricas a codificar: ninguna (ya son todas numéricas)')

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

part = pd.DataFrame({
    'Conjunto': ['Entrenamiento (training)', 'Validación (validation)',
                 'Prueba (test)', 'TOTAL'],
    'Registros': [len(X_train), len(X_valid), len(X_test), len(X)],
    '%': [round(len(X_train)/len(X)*100, 1), round(len(X_valid)/len(X)*100, 1),
          round(len(X_test)/len(X)*100, 1), 100.0]})
print('\n--- TABLA: organización del Data-Set ---')
print(part.to_string(index=False))
part.to_csv(f'{OUT}/tabla_particion.csv', index=False)

bal = pd.DataFrame({
    'Entrenamiento': y_train.map(INV).value_counts(normalize=True).mul(100).round(1),
    'Validación': y_valid.map(INV).value_counts(normalize=True).mul(100).round(1),
    'Prueba': y_test.map(INV).value_counts(normalize=True).mul(100).round(1),
}).reindex(ORDEN)
cnt = pd.DataFrame({
    'Entrenamiento': y_train.map(INV).value_counts(),
    'Validación': y_valid.map(INV).value_counts(),
    'Prueba': y_test.map(INV).value_counts()}).reindex(ORDEN)
print('\n--- Balance de clases (%) ---'); print(bal)
print('\n--- Registros por clase ---'); print(cnt)
bal.to_csv(f'{OUT}/tabla_balance.csv'); cnt.to_csv(f'{OUT}/tabla_conteo_clases.csv')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('\n--- Validación cruzada estratificada (k = 5) sobre entrenamiento ---')
for i, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
    dv = y_train.iloc[va].map(INV).value_counts().reindex(ORDEN).to_dict()
    print(f'  Pliegue {i}: entrena={len(tr):4d}  valida={len(va):4d}   {dv}')

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
b = axes[0].bar(part['Conjunto'][:3], part['Registros'][:3], color=[AZUL, NARANJA, ROJO])
axes[0].set_title('Partición estratificada del Data-Set', fontsize=10)
axes[0].set_ylabel('N.º de registros'); axes[0].tick_params(axis='x', rotation=10, labelsize=8)
for r, v, pc in zip(b, part['Registros'][:3], part['%'][:3]):
    axes[0].text(r.get_x()+r.get_width()/2, v+18, f'{v}\n({pc}%)', ha='center',
                 fontsize=8.5, fontweight='bold')
axes[0].set_ylim(0, part['Registros'][:3].max()*1.2)
bal.plot(kind='bar', ax=axes[1], color=[AZUL, NARANJA, ROJO])
axes[1].set_title('Balance de clases preservado por estratificación', fontsize=10)
axes[1].set_ylabel('% de registros'); axes[1].set_xlabel('Nivel de competitividad')
axes[1].tick_params(axis='x', rotation=0); axes[1].legend(fontsize=8)
fig.savefig(f'{OUT}/fig10_particion.png'); plt.show()

## Celda 9 — Normalización ajustada con entrenamiento (3.2-e)

In [ ]:
num_cols = [c for c in X.columns if X[c].nunique() > 2]
print(f'Variables a escalar: {len(num_cols)}  |  binarias (0/1): {X.shape[1]-len(num_cols)}')

minmax = MinMaxScaler().fit(X_train[num_cols])
zscore = StandardScaler().fit(X_train[num_cols])


def escalar(datos, escalador):
    out = datos.copy()
    out[num_cols] = escalador.transform(datos[num_cols])
    return out


Xtr_mm, Xva_mm, Xte_mm = (escalar(a, minmax) for a in (X_train, X_valid, X_test))
Xtr_z, Xva_z, Xte_z = (escalar(a, zscore) for a in (X_train, X_valid, X_test))

comp = pd.DataFrame({
    'Original_min': X_train[num_cols].min(), 'Original_max': X_train[num_cols].max(),
    'MinMax_min': Xtr_mm[num_cols].min().round(2),
    'MinMax_max': Xtr_mm[num_cols].max().round(2),
    'Zscore_media': Xtr_z[num_cols].mean().round(3),
    'Zscore_desv': Xtr_z[num_cols].std().round(3)})
print('\n--- TABLA: efecto de la normalización (ajustada en entrenamiento) ---')
print(comp.to_string())
comp.to_csv(f'{OUT}/tabla_normalizacion.csv')

print(f'\nRango de validación tras Min-Max: '
      f'[{Xva_mm[num_cols].min().min():.3f} , {Xva_mm[num_cols].max().max():.3f}]')
print(f'Parámetros del escalador tomados de: entrenamiento '
      f'({len(X_train)} registros), no del conjunto completo')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
X_train[num_cols].boxplot(ax=axes[0])
axes[0].set_title('Antes: escalas heterogéneas', fontsize=10)
Xtr_mm[num_cols].boxplot(ax=axes[1])
axes[1].set_title('Después: Min-Max Scaling [0, 1]', fontsize=10)
for a in axes:
    a.tick_params(axis='x', rotation=25, labelsize=8)
fig.suptitle('Efecto de la normalización sobre las variables predictoras',
             y=1.03, fontsize=12, fontweight='bold')
fig.savefig(f'{OUT}/fig9_normalizacion.png'); plt.show()

for nombre, Xs, ys in [('train', Xtr_mm, y_train), ('valid', Xva_mm, y_valid),
                       ('test', Xte_mm, y_test)]:
    Xs.assign(Employment_Class=ys.map(INV)).to_csv(f'{OUT}/dataset_{nombre}.csv', index=False)

print('\n' + '=' * 72)
print('PREPROCESAMIENTO COMPLETADO')
print(f'  Data-Set original : {shape_original[0]} filas x {shape_original[1]} columnas')
print(f'  Data-Set final    : {len(X)} filas x {X.shape[1]} variables predictoras')
print(f'  Variable objetivo : Employment_Class (Low / Medium / High)')
print(f'  Clase mayoritaria : {porc.max():.1f} %  ->  usar F1-macro, no exactitud')
print('=' * 72)
print('Archivos guardados: dataset_train.csv, dataset_valid.csv, dataset_test.csv')

## Celda 10 — Configuración de los modelos (4.2-A)

In [ ]:
MODELOS = {
    'Regresión Logística': {
        'estimador': LogisticRegression(max_iter=2000, class_weight='balanced',
                                        random_state=42),
        'datos': (Xtr_z, Xva_z, Xte_z), 'escalado': 'Z-score'},
    'Árbol de Decisión': {
        'estimador': DecisionTreeClassifier(class_weight='balanced', max_depth=6,
                                            min_samples_leaf=10, random_state=42),
        'datos': (X_train, X_valid, X_test), 'escalado': 'Ninguno'},
    'Random Forest': {
        'estimador': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                            min_samples_leaf=2, random_state=42,
                                            n_jobs=-1),
        'datos': (X_train, X_valid, X_test), 'escalado': 'Ninguno'},
}

print('--- TABLA: configuración de los modelos ---')
for nombre, cfg in MODELOS.items():
    par = cfg['estimador'].get_params()
    clave = {k: par[k] for k in ('max_iter', 'max_depth', 'n_estimators',
                                 'min_samples_leaf') if k in par and par[k] is not None}
    print(f'  {nombre:22s} escalado={cfg["escalado"]:8s} class_weight=balanced  {clave}')

print('\nTodos los modelos usan class_weight="balanced" por el desbalance '
      f'({porc.max():.1f} % de la clase Medium).')

## Celda 11 — Entrenamiento y selección del modelo (4.2-B)

In [ ]:
resultados_cv = {}
print('--- Validación cruzada estratificada (k = 5) sobre entrenamiento ---')
for nombre, cfg in MODELOS.items():
    Xtr = cfg['datos'][0]
    sc = cross_val_score(cfg['estimador'], Xtr, y_train, cv=skf, scoring='f1_macro')
    resultados_cv[nombre] = sc
    print(f'  {nombre:22s} F1-macro = {sc.mean():.4f}  ±{sc.std():.4f}   '
          f'pliegues: {[round(v, 3) for v in sc]}')

print('\n--- Desempeño sobre el conjunto de VALIDACIÓN (300 registros) ---')
filas = []
for nombre, cfg in MODELOS.items():
    Xtr, Xva, _ = cfg['datos']
    mod = cfg['estimador'].fit(Xtr, y_train)
    pred = mod.predict(Xva)
    filas.append({'Modelo': nombre,
                  'CV F1-macro': round(resultados_cv[nombre].mean(), 4),
                  'CV desv.': round(resultados_cv[nombre].std(), 4),
                  'Exactitud': round(accuracy_score(y_valid, pred), 4),
                  'F1-macro': round(f1_score(y_valid, pred, average='macro'), 4),
                  'Precisión': round(precision_score(y_valid, pred, average='macro',
                                                     zero_division=0), 4),
                  'Recall': round(recall_score(y_valid, pred, average='macro'), 4)})
comp_mod = pd.DataFrame(filas).set_index('Modelo')
print(comp_mod.to_string())
comp_mod.to_csv(f'{OUT}/tabla_modelos.csv')

GANADOR = comp_mod['F1-macro'].idxmax()
print(f'\nModelo seleccionado: {GANADOR}  '
      f'(F1-macro = {comp_mod.loc[GANADOR, "F1-macro"]:.4f} en validación)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
nom = list(MODELOS)
axes[0].boxplot([resultados_cv[n] for n in nom], tick_labels=[n.replace(' ', '\n') for n in nom])
axes[0].set_title('Validación cruzada (k = 5) sobre entrenamiento', fontsize=10)
axes[0].set_ylabel('F1-macro')
x = np.arange(len(nom)); an = 0.35
axes[1].bar(x - an/2, comp_mod['Exactitud'], an, label='Exactitud', color=AZUL)
axes[1].bar(x + an/2, comp_mod['F1-macro'], an, label='F1-macro', color=NARANJA)
axes[1].axhline(porc.max()/100, color=ROJO, ls='--', lw=1.2)
axes[1].text(-0.45, porc.max()/100 + .015,
             f'exactitud del clasificador trivial ({porc.max():.1f} %)',
             fontsize=7.5, color=ROJO, ha='left', va='bottom')
axes[1].set_xticks(x); axes[1].set_xticklabels([n.replace(' ', '\n') for n in nom])
axes[1].set_title('Desempeño sobre el conjunto de validación', fontsize=10)
axes[1].set_ylim(0, 1.08); axes[1].legend(fontsize=8)
fig.suptitle('Comparación de los modelos de aprendizaje', y=1.03,
             fontsize=12, fontweight='bold')
fig.savefig(f'{OUT}/fig11_modelos.png'); plt.show()

## Celda 12 — Evaluación final sobre el conjunto de prueba (4.2-C)

In [ ]:
cfg = MODELOS[GANADOR]
Xtr, _, Xte = cfg['datos']
modelo_final = cfg['estimador'].fit(Xtr, y_train)
pred_test = modelo_final.predict(Xte)

print(f'MODELO FINAL: {GANADOR}')
print('\n--- Reporte de clasificación sobre PRUEBA (300 registros) ---')
print(classification_report(y_test, pred_test, target_names=ORDEN, digits=4,
                            zero_division=0))

f1m = f1_score(y_test, pred_test, average='macro')
acc = accuracy_score(y_test, pred_test)
trivial_acc = (y_test == MAPA['Medium']).mean()
print(f'F1-macro  = {f1m:.4f}   |   Exactitud = {acc:.4f}')
print(f'Clasificador trivial (siempre Medium): exactitud = {trivial_acc:.4f}, '
      f'F1-macro = {f1_score(y_test, np.full_like(y_test, MAPA["Medium"]), average="macro", zero_division=0):.4f}')
print(f'RNF-03 (F1-macro >= 0.70): {"CUMPLE" if f1m >= 0.70 else "NO CUMPLE"}')

cm = confusion_matrix(y_test, pred_test)
print('\n--- Matriz de confusión ---')
print(pd.DataFrame(cm, index=[f'real {o}' for o in ORDEN],
                   columns=[f'pred {o}' for o in ORDEN]).to_string())
pd.DataFrame(cm, index=ORDEN, columns=ORDEN).to_csv(f'{OUT}/tabla_confusion.csv')

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
ConfusionMatrixDisplay(cm, display_labels=ORDEN).plot(
    ax=axes[0], cmap='Blues', colorbar=False, values_format='d')
axes[0].set_title('Matriz de confusión (conteos)', fontsize=10)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test, normalize='true'),
                       display_labels=ORDEN).plot(
    ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f')
axes[1].set_title('Normalizada por clase real (recall)', fontsize=10)
for a in axes:
    a.set_xlabel('Predicción'); a.set_ylabel('Clase real')
fig.suptitle(f'Evaluación final sobre el conjunto de prueba — {GANADOR}',
             y=1.04, fontsize=12, fontweight='bold')
fig.savefig(f'{OUT}/fig12_confusion.png'); plt.show()

## Celda 13 — Importancia de las variables (4.2-D)

In [ ]:
rf_fin = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                min_samples_leaf=2, random_state=42,
                                n_jobs=-1).fit(X_train, y_train)
imp_rf = pd.Series(rf_fin.feature_importances_, index=X_train.columns)
coef = pd.DataFrame(modelo_final.coef_, index=ORDEN,
                    columns=Xtr.columns).T if hasattr(modelo_final, 'coef_') else None

print('--- Importancia de variables (Random Forest) ---')
print(imp_rf.sort_values(ascending=False).round(4).to_string())
imp_rf.sort_values(ascending=False).to_csv(f'{OUT}/tabla_importancia.csv')
if coef is not None:
    print('\n--- Coeficientes de la Regresión Logística (por clase) ---')
    print(coef.round(3).to_string())
    coef.to_csv(f'{OUT}/tabla_coeficientes.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
o = imp_rf.sort_values()
axes[0].barh(o.index, o.values, color=AZUL)
axes[0].set_title('Importancia de variables (Random Forest)', fontsize=10)
axes[0].set_xlabel('Importancia relativa')
for i, v in enumerate(o.values):
    axes[0].text(v + .004, i, f'{v:.3f}', va='center', fontsize=8)
axes[0].set_xlim(0, o.max() * 1.18)

if coef is not None:
    coef.loc[o.index].plot(kind='barh', ax=axes[1], color=[ROJO, NARANJA, VERDE],
                           legend=False)
    axes[1].axvline(0, color='#555', lw=.9)
    axes[1].set_title('Coeficientes de la Regresión Logística', fontsize=10)
    axes[1].set_xlabel('Peso (datos estandarizados)')
    axes[1].set_yticklabels([])
    axes[1].set_ylabel('')
    axes[1].legend(title='Clase', fontsize=8, title_fontsize=8, loc='lower right')
fig.subplots_adjust(wspace=0.08)
fig.suptitle('Variables que más determinan la competitividad laboral',
             y=1.04, fontsize=12, fontweight='bold')
fig.savefig(f'{OUT}/fig13_importancia.png'); plt.show()

print('\n' + '=' * 72)
print(f'APRENDIZAJE COMPLETADO  |  modelo: {GANADOR}  |  F1-macro (prueba) = {f1m:.4f}')
print('=' * 72)

# Secciones 4.2-E y 5 — Red neuronal (Deep Learning) y comprobación

## Celda 14 — Preparación de los datos para la red neuronal (5.1-a)

In [ ]:
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 170, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10})
AZUL, NARANJA, GRIS = '#2F5D8A', '#D9822B', '#8A9199'
VARS = ['GPA', 'Programming_Skill', 'Communication_Skill', 'Teamwork',
        'Internship_Count', 'Industry_Exposure']
MAPA = {'Low': 0, 'Medium': 1, 'High': 2}
CLASES = ['Low', 'Medium', 'High']
X = df[VARS]
y = df['Employment_Class'].map(MAPA)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

escalador = StandardScaler().fit(X_train)
Xtr, Xva, Xte = (escalador.transform(d) for d in (X_train, X_valid, X_test))
ytr, yva, yte = y_train.to_numpy(), y_valid.to_numpy(), y_test.to_numpy()

for nombre, ys in [('Entrenamiento', ytr), ('Validación', yva), ('Prueba', yte)]:
    cuenta = np.bincount(ys, minlength=3)
    print(f'{nombre:<14} {len(ys):>5} registros | Low {cuenta[0]:>3} | '
          f'Medium {cuenta[1]:>4} | High {cuenta[2]:>3}')


def metricas(y_real, y_pred):
    return {'Exactitud': accuracy_score(y_real, y_pred),
            'F1-macro': f1_score(y_real, y_pred, average='macro'),
            'Precisión': precision_score(y_real, y_pred, average='macro', zero_division=0),
            'Recall': recall_score(y_real, y_pred, average='macro')}

## Celda 15 — Modelo de referencia: Regresión Logística (5.1-a)

In [ ]:
reg_log = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
reg_log.fit(Xtr, ytr)
m_rl_val = metricas(yva, reg_log.predict(Xva))
print('Regresión Logística (modelo de Machine Learning seleccionado en el punto 4)')
print('  Validación:', {k: round(v, 4) for k, v in m_rl_val.items()})

## Celda 16 — Arquitectura de la red neuronal (4.2-E)

In [ ]:
CAPAS = (32, 16)


def construir_red(semilla=42):
    return MLPClassifier(hidden_layer_sizes=CAPAS, activation='relu', solver='adam',
                         learning_rate_init=0.001, alpha=0.001, batch_size=32,
                         max_iter=1, random_state=semilla)


n_param = sum(a * b + b for a, b in zip((len(VARS),) + CAPAS, CAPAS + (3,)))
print('Arquitectura de la red neuronal')
print(f'  Capa de entrada : {len(VARS)} neuronas (variables estandarizadas)')
print(f'  Capa oculta 1   : {CAPAS[0]} neuronas, activación ReLU')
print(f'  Capa oculta 2   : {CAPAS[1]} neuronas, activación ReLU')
print('  Capa de salida  : 3 neuronas, activación softmax')
print('  Optimizador Adam (tasa 0.001), regularización L2 = 0.001, lotes de 32')
print(f'  Parámetros entrenables: {n_param}')

fig, ax = plt.subplots(figsize=(8.6, 4.4))
ax.axis('off')
capas_dib = [(len(VARS), 'Entrada\n6 variables'), (12, 'Oculta 1\n32 ReLU'),
             (8, 'Oculta 2\n16 ReLU'), (3, 'Salida\n3 softmax')]
xs = [0.08, 0.36, 0.64, 0.92]
pos = []
for (n, _), x in zip(capas_dib, xs):
    ys_ = np.linspace(0.85, 0.15, n) if n > 1 else [0.5]
    pos.append([(x, yy) for yy in ys_])
for a, b in zip(pos, pos[1:]):
    for p in a:
        for q in b:
            ax.plot([p[0], q[0]], [p[1], q[1]], color='#C9CED3', lw=0.5, zorder=1)
colores = [AZUL, '#5B8466', '#5B8466', NARANJA]
for capa, (n, etiqueta), x, c in zip(pos, capas_dib, xs, colores):
    for (px, py) in capa:
        ax.scatter([px], [py], s=150, color=c, zorder=2)
    ax.text(x, 0.02, etiqueta, ha='center', va='top', fontsize=9.5)
for (px, py), nom in zip(pos[0], VARS):
    ax.text(px - 0.035, py, nom, ha='right', va='center', fontsize=8)
for (px, py), nom in zip(pos[3], CLASES):
    ax.text(px + 0.035, py, nom, ha='left', va='center', fontsize=9)
ax.text(0.36, 0.93, '(se dibujan 12 de 32)', ha='center', fontsize=7.5, color=GRIS)
ax.text(0.64, 0.93, '(se dibujan 8 de 16)', ha='center', fontsize=7.5, color=GRIS)
ax.set_xlim(-0.12, 1.08); ax.set_ylim(-0.12, 1.0)
ax.set_title('Arquitectura de la red neuronal multicapa (MLP)', fontsize=11.5)
fig.savefig(f'{OUT}/fig_s9_arquitectura.png'); plt.show()

## Celda 17 — Entrenamiento por épocas con parada temprana (5.1-b)

In [ ]:
pesos = compute_sample_weight('balanced', ytr)
red = construir_red()
historial = {'epoca': [], 'loss_train': [], 'loss_valid': [], 'f1_train': [], 'f1_valid': []}
mejor_f1, mejor_red, mejor_epoca, sin_mejora = -1, None, 0, 0
PACIENCIA, MAX_EPOCAS = 25, 300
for epoca in range(1, MAX_EPOCAS + 1):
    red.partial_fit(Xtr, ytr, classes=np.array([0, 1, 2]), sample_weight=pesos)
    historial['epoca'].append(epoca)
    historial['loss_train'].append(log_loss(ytr, red.predict_proba(Xtr), sample_weight=pesos))
    historial['loss_valid'].append(log_loss(yva, red.predict_proba(Xva),
                                            sample_weight=compute_sample_weight('balanced', yva)))
    historial['f1_train'].append(f1_score(ytr, red.predict(Xtr), average='macro'))
    f1v = f1_score(yva, red.predict(Xva), average='macro')
    historial['f1_valid'].append(f1v)
    if f1v > mejor_f1 + 1e-4:
        mejor_f1, mejor_red, mejor_epoca, sin_mejora = f1v, copy.deepcopy(red), epoca, 0
    else:
        sin_mejora += 1
    if sin_mejora >= PACIENCIA:
        break
red = mejor_red
hist = pd.DataFrame(historial)
print(f'Entrenamiento detenido en la época {epoca} (paciencia = {PACIENCIA}).')
print(f'Mejor época: {mejor_epoca} | F1-macro en validación = {mejor_f1:.4f}')
print(hist.iloc[[0, 4, 9, 19, mejor_epoca - 1, len(hist) - 1]].round(4).to_string(index=False))

fig, axs = plt.subplots(1, 2, figsize=(11, 3.9))
axs[0].plot(hist.epoca, hist.loss_train, color=AZUL, label='Entrenamiento')
axs[0].plot(hist.epoca, hist.loss_valid, color=NARANJA, label='Validación')
axs[0].set_title('Función de pérdida (entropía cruzada ponderada)')
axs[0].set_xlabel('Época'); axs[0].set_ylabel('Pérdida')
axs[1].plot(hist.epoca, hist.f1_train, color=AZUL, label='Entrenamiento')
axs[1].plot(hist.epoca, hist.f1_valid, color=NARANJA, label='Validación')
axs[1].legend(frameon=False, loc='lower right')
axs[1].set_title('F1-macro por época'); axs[1].set_xlabel('Época'); axs[1].set_ylabel('F1-macro')
for a in axs:
    a.axvline(mejor_epoca, color=GRIS, ls='--', lw=1)
axs[0].legend(frameon=False)
axs[1].text(mejor_epoca - 2, 0.55, f'mejor época = {mejor_epoca}',
            color=GRIS, fontsize=9, ha='right')
fig.suptitle('Historial del entrenamiento de la red neuronal', fontsize=12, y=1.04)
fig.savefig(f'{OUT}/fig_s9_historial.png'); plt.show()

## Celda 18 — Validación cruzada estratificada (5.1-c)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = {'Regresión Logística': [], 'Red neuronal (MLP)': []}
Xtr_crudo = X_train.to_numpy()
for k, (i_tr, i_va) in enumerate(skf.split(Xtr_crudo, ytr), 1):
    esc_k = StandardScaler().fit(Xtr_crudo[i_tr])
    a, b = esc_k.transform(Xtr_crudo[i_tr]), esc_k.transform(Xtr_crudo[i_va])
    rl = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42).fit(a, ytr[i_tr])
    cv['Regresión Logística'].append(f1_score(ytr[i_va], rl.predict(b), average='macro'))
    rk = construir_red()
    w_k = compute_sample_weight('balanced', ytr[i_tr])
    for _ in range(mejor_epoca):
        rk.partial_fit(a, ytr[i_tr], classes=np.array([0, 1, 2]), sample_weight=w_k)
    cv['Red neuronal (MLP)'].append(f1_score(ytr[i_va], rk.predict(b), average='macro'))
cv_df = pd.DataFrame(cv, index=[f'Pliegue {i}' for i in range(1, 6)])
print(cv_df.round(4).to_string())
print('Media  ', '  '.join(f'{c}: {cv_df[c].mean():.4f} ± {cv_df[c].std(ddof=0):.4f}'
                            for c in cv_df.columns))

## Celda 19 — Comparación sobre validación y selección (5.1-d)

In [ ]:
m_red_val = metricas(yva, red.predict(Xva))
comp = pd.DataFrame({'Regresión Logística': m_rl_val, 'Red neuronal (MLP)': m_red_val}).T
comp.insert(0, 'CV F1-macro', [cv_df[c].mean() for c in comp.index])
comp.insert(1, 'Desv. CV', [cv_df[c].std(ddof=0) for c in comp.index])
print(comp.round(4).to_string())
seleccion = comp['F1-macro'].idxmax() if abs(comp['F1-macro'].diff().iloc[-1]) > 0.01 \
    else comp['Desv. CV'].idxmin()
print('\nModelo seleccionado para el sistema:', seleccion)
comp.to_csv(f'{OUT}/tabla_s9_comparacion.csv')

## Celda 20 — Aplicación a los Data-Sets de entrenamiento y de prueba (5.1-e)

In [ ]:
filas = []
for nombre, modelo in [('Regresión Logística', reg_log), ('Red neuronal (MLP)', red)]:
    for conjunto, Xs, ys in [('Entrenamiento', Xtr, ytr), ('Prueba', Xte, yte)]:
        filas.append({'Modelo': nombre, 'Conjunto': conjunto, **metricas(ys, modelo.predict(Xs))})
comprob = pd.DataFrame(filas)
print(comprob.round(4).to_string(index=False))
comprob.to_csv(f'{OUT}/tabla_s9_comprobacion.csv', index=False)
print('\nReporte por clase de la red neuronal sobre el conjunto de prueba:')
print(classification_report(yte, red.predict(Xte), target_names=CLASES, digits=4))

fig, axs = plt.subplots(1, 2, figsize=(10.4, 4.1))
for a, (nombre, modelo) in zip(axs, [('Regresión Logística', reg_log), ('Red neuronal (MLP)', red)]):
    cm = confusion_matrix(yte, modelo.predict(Xte))
    a.imshow(cm, cmap='Blues')
    for i in range(3):
        for j in range(3):
            a.text(j, i, cm[i, j], ha='center', va='center',
                   color='white' if cm[i, j] > cm.max() / 2 else '#1E2B38', fontsize=12)
    a.set_xticks(range(3)); a.set_xticklabels(CLASES); a.set_yticks(range(3)); a.set_yticklabels(CLASES)
    a.set_xlabel('Predicción'); a.set_ylabel('Clase real'); a.set_title(nombre)
    for s in a.spines.values():
        s.set_visible(False)
fig.suptitle('Matrices de confusión sobre el conjunto de prueba (300 estudiantes)', fontsize=12, y=1.04)
fig.savefig(f'{OUT}/fig_s9_confusion.png'); plt.show()

## Celda 21 — Resumen gráfico de la comprobación (5.1-e)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.9))
etiquetas = ['Exactitud', 'F1-macro', 'Precisión', 'Recall']
ancho = 0.2
for i, (nombre, conj, color, alfa) in enumerate([
        ('Regresión Logística', 'Entrenamiento', AZUL, 0.45), ('Regresión Logística', 'Prueba', AZUL, 1),
        ('Red neuronal (MLP)', 'Entrenamiento', NARANJA, 0.45), ('Red neuronal (MLP)', 'Prueba', NARANJA, 1)]):
    fila = comprob[(comprob.Modelo == nombre) & (comprob.Conjunto == conj)].iloc[0]
    vals = [fila[e] for e in etiquetas]
    barras = ax.bar(np.arange(4) + (i - 1.5) * ancho, vals, ancho, color=color, alpha=alfa,
                    label=f'{nombre} · {conj.lower()}')
    for b_, v in zip(barras, vals):
        ax.text(b_.get_x() + b_.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center', fontsize=7)
ax.axhline(0.70, color=GRIS, ls='--', lw=1)
ax.text(3.52, 0.70, 'umbral\nRNF-03\n(0.70)', color=GRIS, fontsize=8, ha='left', va='center')
ax.set_xlim(-0.5, 4.0)
ax.set_xticks(range(4)); ax.set_xticklabels(etiquetas); ax.set_ylim(0, 1.12)
ax.legend(frameon=False, fontsize=8, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax.set_title('Comprobación: desempeño en entrenamiento y en prueba')
fig.savefig(f'{OUT}/fig_s9_comprobacion.png'); plt.show()
print(f"F1-macro en prueba -> Regresión Logística: "
      f"{comprob.query('Conjunto==\"Prueba\" and Modelo==\"Regresión Logística\"')['F1-macro'].iloc[0]:.4f}"
      f" | Red neuronal: {comprob.query('Conjunto==\"Prueba\" and Modelo==\"Red neuronal (MLP)\"')['F1-macro'].iloc[0]:.4f}")

# Sección 6 — Evaluación

## Celda 22 — Datos, partición y entrenamiento de los cuatro modelos (6.1)

In [ ]:
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 170, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
AZUL, NARANJA, VERDE, GRIS = '#2F5D8A', '#D9822B', '#5B8466', '#8A9199'
VARS = ['GPA', 'Programming_Skill', 'Communication_Skill', 'Teamwork',
        'Internship_Count', 'Industry_Exposure']
CLASES = ['Low', 'Medium', 'High']
X = df[VARS]
y = df['Employment_Class'].map({'Low': 0, 'Medium': 1, 'High': 2})


def particionar(semilla):
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=semilla, stratify=y)
    X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=semilla, stratify=y_tmp)
    return X_tr, X_va, X_te, y_tr.to_numpy(), y_va.to_numpy(), y_te.to_numpy()


def construir_red(semilla=42):
    return MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', solver='adam',
                         learning_rate_init=0.001, alpha=0.001, batch_size=32,
                         max_iter=1, random_state=semilla)


def entrenar_red(Xtr, ytr, Xva, yva, semilla=42, paciencia=25, max_epocas=300):
    pesos = compute_sample_weight('balanced', ytr)
    red = construir_red(semilla)
    mejor, mejor_red, mejor_epoca, espera = -1, None, 0, 0
    for epoca in range(1, max_epocas + 1):
        red.partial_fit(Xtr, ytr, classes=np.array([0, 1, 2]), sample_weight=pesos)
        f1 = f1_score(yva, red.predict(Xva), average='macro')
        if f1 > mejor + 1e-4:
            mejor, mejor_red, mejor_epoca, espera = f1, copy.deepcopy(red), epoca, 0
        else:
            espera += 1
        if espera >= paciencia:
            break
    return mejor_red, mejor_epoca


def entrenar_todos(semilla=42):
    X_tr, X_va, X_te, ytr, yva, yte = particionar(semilla)
    esc = StandardScaler().fit(X_tr)
    reg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=semilla)
    arbol = DecisionTreeClassifier(max_depth=6, min_samples_leaf=10,
                                   class_weight='balanced', random_state=semilla)
    bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                    class_weight='balanced', random_state=semilla)
    reg.fit(esc.transform(X_tr), ytr)
    arbol.fit(X_tr, ytr)
    bosque.fit(X_tr, ytr)
    red, epoca = entrenar_red(esc.transform(X_tr), ytr, esc.transform(X_va), yva, semilla)
    sistemas = {'Regresión Logística': Pipeline([('esc', esc), ('modelo', reg)]),
                'Árbol de Decisión': Pipeline([('esc', 'passthrough'), ('modelo', arbol)]),
                'Random Forest': Pipeline([('esc', 'passthrough'), ('modelo', bosque)]),
                'Red neuronal (MLP)': Pipeline([('esc', esc), ('modelo', red)])}
    return sistemas, epoca, (X_tr, X_va, X_te, ytr, yva, yte)


SISTEMAS, EPOCA, (X_tr, X_va, X_te, ytr, yva, yte) = entrenar_todos(42)
for nombre, ys in [('Entrenamiento', ytr), ('Validación', yva), ('Prueba', yte)]:
    c = np.bincount(ys, minlength=3)
    print(f'{nombre:<14}{len(ys):>5} registros | Low {c[0]:>3} | Medium {c[1]:>4} | High {c[2]:>3}')
print('Modelos entrenados:', ', '.join(SISTEMAS))
print('Mejor época de la red neuronal:', EPOCA)

## Celda 23 — Métricas globales en el conjunto de prueba (6.1-a)

In [ ]:
def resumen(sistema, Xs, ys):
    pred = sistema.predict(Xs)
    prob = sistema.predict_proba(Xs)
    return {'Exactitud': accuracy_score(ys, pred),
            'F1-macro': f1_score(ys, pred, average='macro'),
            'Precisión': precision_score(ys, pred, average='macro', zero_division=0),
            'Recall': recall_score(ys, pred, average='macro'),
            'MCC': matthews_corrcoef(ys, pred),
            'AUC-ROC': roc_auc_score(ys, prob, multi_class='ovr', average='macro')}


metricas_prueba = pd.DataFrame({n: resumen(s, X_te, yte) for n, s in SISTEMAS.items()}).T
print('Métricas sobre el conjunto de prueba (300 estudiantes)')
print(metricas_prueba.round(4).to_string())
metricas_prueba.to_csv(f'{OUT}/tabla_s11_metricas_prueba.csv')

## Celda 24 — Métricas por clase y matrices de confusión (6.1-b)

In [ ]:
FINALISTAS = ['Regresión Logística', 'Red neuronal (MLP)']
for nombre in FINALISTAS:
    print(nombre)
    print(classification_report(yte, SISTEMAS[nombre].predict(X_te), target_names=CLASES,
                                digits=4))

fig, axs = plt.subplots(1, 2, figsize=(10.6, 4.3))
for a, nombre in zip(axs, FINALISTAS):
    cm = confusion_matrix(yte, SISTEMAS[nombre].predict(X_te))
    pct = cm / cm.sum(axis=1, keepdims=True)
    a.imshow(pct, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            texto = f'{pct[i, j]:.1%}' if 0 < pct[i, j] < 0.01 else f'{pct[i, j]:.0%}'
            a.text(j, i, f'{cm[i, j]}\n({texto})', ha='center', va='center',
                   color='white' if pct[i, j] > 0.5 else '#1E2B38', fontsize=11)
    a.set_xticks(range(3)); a.set_xticklabels(CLASES)
    a.set_yticks(range(3)); a.set_yticklabels(CLASES)
    a.set_xlabel('Predicción'); a.set_ylabel('Clase real'); a.set_title(nombre)
    for borde in a.spines.values():
        borde.set_visible(False)
fig.suptitle('Matrices de confusión en el conjunto de prueba', fontsize=12, y=1.04)
fig.savefig(f'{OUT}/fig_s11_confusion.png'); plt.show()

## Celda 25 — Curvas ROC y precisión–recall (6.1-c)

In [ ]:
prob = SISTEMAS['Red neuronal (MLP)'].predict_proba(X_te)
colores = [NARANJA, AZUL, VERDE]
fig, axs = plt.subplots(1, 2, figsize=(10.6, 4.3))
for k, clase in enumerate(CLASES):
    real = (yte == k).astype(int)
    fpr, tpr, _ = roc_curve(real, prob[:, k])
    pre, rec, _ = precision_recall_curve(real, prob[:, k])
    auc, ap = roc_auc_score(real, prob[:, k]), average_precision_score(real, prob[:, k])
    axs[0].plot(fpr, tpr, color=colores[k], label=f'{clase} (AUC = {auc:.3f})')
    axs[1].plot(rec, pre, color=colores[k], label=f'{clase} (AP = {ap:.3f})')
    print(f'{clase:<7} AUC-ROC = {auc:.4f} | precisión promedio = {ap:.4f}')
axs[0].plot([0, 1], [0, 1], color=GRIS, ls='--', lw=1)
axs[0].set_xlabel('Tasa de falsos positivos'); axs[0].set_ylabel('Tasa de verdaderos positivos')
axs[0].set_title('Curvas ROC (una clase frente al resto)')
axs[1].set_xlabel('Recall'); axs[1].set_ylabel('Precisión')
axs[1].set_title('Curvas precisión–recall')
for a in axs:
    a.legend(frameon=False, loc='lower right' if a is axs[0] else 'lower left')
fig.savefig(f'{OUT}/fig_s11_roc_pr.png'); plt.show()

## Celda 26 — Análisis de errores y regla de etiquetado (6.1-d y 6.1-e)

In [ ]:
formula = LinearRegression().fit(X, df['Employment_Competitiveness'])
r2 = formula.score(X, df['Employment_Competitiveness'])
print(f'R² del puntaje explicado por las seis variables: {r2:.6f}')
print(pd.Series(formula.coef_, index=VARS).round(4).to_string())
print(f'Intercepto: {formula.intercept_:.4f}\n')
pred = SISTEMAS['Red neuronal (MLP)'].predict(X_te)
prob = SISTEMAS['Red neuronal (MLP)'].predict_proba(X_te)
print(df.groupby('Employment_Class')['Employment_Competitiveness'].agg(['min', 'max']).round(2))
analisis = pd.DataFrame({'Real': [CLASES[i] for i in yte],
                         'Predicho': [CLASES[i] for i in pred],
                         'Puntaje': df.loc[X_te.index, 'Employment_Competitiveness'].to_numpy(),
                         'Prob_max': prob.max(axis=1)}, index=X_te.index)
analisis['Dist_umbral'] = np.minimum((analisis['Puntaje'] - 40).abs(),
                                     (analisis['Puntaje'] - 70).abs())
analisis['Error'] = analisis['Real'] != analisis['Predicho']
errores = analisis[analisis['Error']].sort_values('Dist_umbral')
print(f'\nErrores de la red neuronal: {len(errores)} de {len(analisis)}')
print(errores.drop(columns='Error').round(3).to_string())
aciertos = analisis[~analisis['Error']]
print(f"\nDistancia mediana al umbral más cercano: errores {errores['Dist_umbral'].median():.2f}"
      f" | aciertos {aciertos['Dist_umbral'].median():.2f}")
print(f"Errores a menos de 3 puntos de un umbral: {(errores['Dist_umbral'] < 3).sum()}"
      f" de {len(errores)}")
errores.drop(columns='Error').to_csv(f'{OUT}/tabla_s11_errores.csv')

rng = np.random.default_rng(7)
fig, ax = plt.subplots(figsize=(9.6, 3.6))
for ok, color, etiqueta in [(True, GRIS, 'Acierto'), (False, NARANJA, 'Error')]:
    sub = analisis[analisis['Error'] != ok]
    ax.scatter(sub['Puntaje'], rng.uniform(-0.3, 0.3, len(sub)), s=18 if ok else 34,
               color=color, alpha=0.55 if ok else 0.95, label=f'{etiqueta} ({len(sub)})')
for umbral in (40, 70):
    ax.axvline(umbral, color=AZUL, ls='--', lw=1)
    ax.text(umbral + 0.5, 0.42, f'umbral {umbral}', color=AZUL, fontsize=9)
ax.set_yticks([]); ax.set_ylim(-0.5, 0.55)
ax.set_xlabel('Puntaje de competitividad del dataset (Employment_Competitiveness)')
ax.set_title('Ubicación de los aciertos y errores respecto de los umbrales de clase')
ax.legend(frameon=False, loc='upper center', ncol=2, bbox_to_anchor=(0.5, -0.22))
fig.savefig(f'{OUT}/fig_s11_errores.png'); plt.show()

## Celda 27 — Estabilidad con 10 particiones (6.2-b)

In [ ]:
filas = []
for semilla in range(10):
    X_a, X_b, X_c, y_a, y_b, y_c = particionar(semilla)
    esc = StandardScaler().fit(X_a)
    a, b, c = esc.transform(X_a), esc.transform(X_b), esc.transform(X_c)
    reg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=semilla)
    reg.fit(a, y_a)
    red, epoca = entrenar_red(a, y_a, b, y_b, semilla)
    filas.append({'Semilla': semilla, 'Época': epoca,
                  'Regresión Logística': f1_score(y_c, reg.predict(c), average='macro'),
                  'Red neuronal (MLP)': f1_score(y_c, red.predict(c), average='macro')})
estabilidad = pd.DataFrame(filas)
print('F1-macro en prueba con 10 particiones distintas')
print(estabilidad.round(4).to_string(index=False))
resumen_estab = estabilidad[FINALISTAS].agg(['mean', 'std', 'min', 'max']).T
print()
print(resumen_estab.round(4).to_string())
print('\nParticiones con F1-macro >= 0.70:',
      {n: int((estabilidad[n] >= 0.70).sum()) for n in FINALISTAS}, 'de 10')
estabilidad.to_csv(f'{OUT}/tabla_s11_estabilidad.csv', index=False)

## Celda 28 — Robustez ante errores de medición (6.2-c)

In [ ]:
niveles = [0, 0.1, 0.25, 0.5, 1.0]
desv, minimo, maximo = X_tr.std().to_numpy(), X_tr.min().to_numpy(), X_tr.max().to_numpy()
rng = np.random.default_rng(42)
robustez = {n: [] for n in FINALISTAS}
for nivel in niveles:
    acum = {n: [] for n in FINALISTAS}
    for _ in range(20):
        ruido = rng.normal(0, 1, X_te.shape) * desv * nivel
        X_ruido = pd.DataFrame(np.clip(X_te.to_numpy() + ruido, minimo, maximo), columns=VARS)
        for n in FINALISTAS:
            acum[n].append(f1_score(yte, SISTEMAS[n].predict(X_ruido), average='macro'))
    for n in FINALISTAS:
        robustez[n].append(np.mean(acum[n]))
tabla_robustez = pd.DataFrame(robustez, index=[f'{int(v * 100)} %' for v in niveles])
tabla_robustez.index.name = 'Ruido (% de la desv. típica)'
print('F1-macro en prueba al perturbar las seis variables (promedio de 20 repeticiones)')
print(tabla_robustez.round(4).to_string())
tabla_robustez.to_csv(f'{OUT}/tabla_s11_robustez.csv')

fig, ax = plt.subplots(figsize=(8.4, 3.9))
for n, color in zip(FINALISTAS, [AZUL, NARANJA]):
    ax.plot([v * 100 for v in niveles], robustez[n], marker='o', color=color, label=n)
ax.axhline(0.70, color=GRIS, ls='--', lw=1)
ax.text(101, 0.72, 'umbral\nRNF-03', color=GRIS, fontsize=8, va='bottom')
ax.set_xlabel('Ruido añadido (% de la desviación típica de cada variable)')
ax.set_ylabel('F1-macro en prueba')
ax.set_title('Robustez del modelo ante errores de medición')
ax.set_xlim(-3, 112); ax.set_ylim(0, 1)
ax.legend(frameon=False, loc='lower left')
fig.savefig(f'{OUT}/fig_s11_robustez.png'); plt.show()

## Celda 29 — Equidad por grupos (6.2-d)

In [ ]:
Xn, yn = X.to_numpy(), y.to_numpy()
cruzadas = {n: np.zeros(len(yn), dtype=int) for n in FINALISTAS}
for i_tr, i_te in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xn, yn):
    esc = StandardScaler().fit(Xn[i_tr])
    a, b = esc.transform(Xn[i_tr]), esc.transform(Xn[i_te])
    reg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
    cruzadas['Regresión Logística'][i_te] = reg.fit(a, yn[i_tr]).predict(b)
    red, pesos = construir_red(), compute_sample_weight('balanced', yn[i_tr])
    for _ in range(EPOCA):
        red.partial_fit(a, yn[i_tr], classes=np.array([0, 1, 2]), sample_weight=pesos)
    cruzadas['Red neuronal (MLP)'][i_te] = red.predict(b)


def por_grupo(pred, columna):
    filas = []
    for grupo, idx in df.groupby(columna).indices.items():
        real, p = yn[idx], pred[idx]
        filas.append({'Variable': columna, 'Grupo': grupo, 'N': len(idx),
                      'F1-macro': f1_score(real, p, average='macro'),
                      'Recall Low': recall_score(real == 0, p == 0),
                      'Recall High': recall_score(real == 2, p == 2),
                      '% High real': 100 * np.mean(real == 2),
                      '% High predicho': 100 * np.mean(p == 2)})
    return filas


equidad = {}
for n in FINALISTAS:
    tabla = pd.DataFrame(por_grupo(cruzadas[n], 'Gender') + por_grupo(cruzadas[n], 'Degree_Level'))
    equidad[n] = tabla
    print(n, '(predicciones de validación cruzada, 2 000 estudiantes)')
    print(tabla.round(3).to_string(index=False))
    for var in ('Gender', 'Degree_Level'):
        f1s = tabla.loc[tabla['Variable'] == var, 'F1-macro']
        print(f'  Brecha máxima de F1-macro entre grupos de {var}: {f1s.max() - f1s.min():.4f}')
    print()
equidad['Red neuronal (MLP)'].to_csv(f'{OUT}/tabla_s11_equidad.csv', index=False)

tabla = equidad['Red neuronal (MLP)']
etiquetas = [f"{'Género' if v == 'Gender' else 'Nivel'} {g}"
             for v, g in zip(tabla['Variable'], tabla['Grupo'])]
fig, ax = plt.subplots(figsize=(9.4, 3.9))
pos = np.arange(len(tabla))
ax.bar(pos - 0.2, tabla['F1-macro'], 0.4, color=AZUL, label='F1-macro')
ax.bar(pos + 0.2, tabla['Recall High'], 0.4, color=NARANJA, label='Recall de la clase High')
for p_, v1, v2 in zip(pos, tabla['F1-macro'], tabla['Recall High']):
    ax.text(p_ - 0.2, v1 + 0.01, f'{v1:.2f}', ha='center', fontsize=8)
    ax.text(p_ + 0.2, v2 + 0.01, f'{v2:.2f}', ha='center', fontsize=8)
ax.set_xticks(pos); ax.set_xticklabels(etiquetas); ax.set_ylim(0, 1.15)
ax.set_title('Desempeño de la red neuronal por grupo (variables no usadas como entrada)')
ax.legend(frameon=False, loc='upper center', ncol=2, bbox_to_anchor=(0.5, -0.1))
fig.savefig(f'{OUT}/fig_s11_equidad.png'); plt.show()

## Celda 30 — Desempeño operativo y verificación de requisitos (6.2-e y 6.2-f)

In [ ]:
modelo = SISTEMAS['Red neuronal (MLP)']
t0 = time.perf_counter()
entrenar_todos(42)
seg_entrenar = time.perf_counter() - t0

fila = X_te.iloc[[0]]
ms = []
for _ in range(500):
    t0 = time.perf_counter()
    modelo.predict_proba(fila)
    ms.append((time.perf_counter() - t0) * 1000)
print(f'Reentrenar los cuatro modelos completos: {seg_entrenar:.1f} s')
print(f'Predicción de un estudiante: mediana {np.median(ms):.2f} ms | '
      f'percentil 95 {np.percentile(ms, 95):.2f} ms | máximo {max(ms):.2f} ms')

escalado = []
for n in (1000, 10000, 100000):
    lote = pd.concat([X_te] * (n // len(X_te) + 1)).iloc[:n]
    t0 = time.perf_counter()
    modelo.predict(lote)
    seg = time.perf_counter() - t0
    escalado.append({'Registros': n, 'Segundos': seg, 'ms por 1 000 registros': seg / n * 1e6})
escalado = pd.DataFrame(escalado)
print()
print(escalado.round(4).to_string(index=False))

print()
for n, s in SISTEMAS.items():
    print(f'Tamaño en disco de {n}: {len(pickle.dumps(s)) / 1024:.1f} KB')

f1_prueba = metricas_prueba.loc['Red neuronal (MLP)', 'F1-macro']
requisitos = pd.DataFrame([
    ('RNF-02', 'Predicción < 5 s por estudiante', f'{np.percentile(ms, 95) / 1000:.4f} s (p95)',
     np.percentile(ms, 95) / 1000 < 5),
    ('RNF-02', 'Reentrenamiento completo < 10 min', f'{seg_entrenar:.1f} s', seg_entrenar < 600),
    ('RNF-03', 'F1-macro en prueba >= 0.70', f'{f1_prueba:.4f}', f1_prueba >= 0.70),
    ('RNF-03', 'Desempeño consistente (10 particiones)',
     f"mín. {estabilidad['Red neuronal (MLP)'].min():.4f}, "
     f"desv. {estabilidad['Red neuronal (MLP)'].std():.4f}",
     estabilidad['Red neuronal (MLP)'].min() >= 0.70),
    ('RNF-04', '10 000 registros sin degradación',
     f"{escalado.loc[escalado['Registros'] == 10000, 'Segundos'].iloc[0]:.3f} s",
     escalado['ms por 1 000 registros'].iloc[1] <= escalado['ms por 1 000 registros'].iloc[0]),
], columns=['Código', 'Criterio', 'Valor medido', 'Cumple'])
print()
print(requisitos.to_string(index=False))
requisitos.to_csv(f'{OUT}/tabla_s11_requisitos.csv', index=False)